In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.isdir(os.path.join(_root, "tools")) and os.path.isdir(os.path.join(_root, "data")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from tools.paths import data_path, outputs_path

In [ ]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# LOAD — clean Airtable export, no header/data offset issue this time.
# ─────────────────────────────────────────────────────────────────────────────
rematch = pd.read_csv(data_path('Student Placement Confirmation Form-Rematch request.csv'), index_col=False)

print("Total rematch request rows:", len(rematch))

# ─────────────────────────────────────────────────────────────────────────────
# EXTRACT STUDENT CODE — EMPLID column is entirely blank in this export, but
# the ID is embedded in "Application ID: Full Name | EMPLID (from Students)",
# e.g. "27467: Leona Sahajalal | 24619760". Pull the number after the pipe.
# ─────────────────────────────────────────────────────────────────────────────
id_col = 'Application ID: Full Name | EMPLID (from Students)'
rematch['Student Code'] = rematch[id_col].str.extract(r'\|\s*(\d+)')[0].astype(int)

unique_students = rematch['Student Code'].nunique()
repeat_counts = rematch.groupby('Student Code').size()
repeat_requesters = (repeat_counts > 1).sum()

print(f"Unique students who submitted a rematch request: {unique_students}")
print(f"Students who submitted more than one request: {repeat_requesters}")
print(f"Max requests from a single student: {repeat_counts.max()}")

# ─────────────────────────────────────────────────────────────────────────────
# REMOVE DUPLICATES — the "Notes" column flags rows staff had already marked
# as duplicate submissions (e.g. "DUPLICATE", "duplicate rematch request",
# or in one case a student explicitly withdrawing a repeat request). These
# are redundant log entries for the same underlying request, not separate
# rematch events, so they need to come out before counting anything.
# ─────────────────────────────────────────────────────────────────────────────
is_duplicate = rematch['Notes'].str.contains('duplicate', case=False, na=False)
print(f"\nRows flagged as duplicate in Notes: {is_duplicate.sum()}")
rematch = rematch[~is_duplicate].copy()
print(f"Clean rematch requests after removing duplicates: {len(rematch)}")
print(f"Clean unique students: {rematch['Student Code'].nunique()}")

clean_repeat_counts = rematch.groupby('Student Code').size()
print(f"Students with >1 CLEAN request: {(clean_repeat_counts > 1).sum()}")
print(f"Max clean requests from a single student: {clean_repeat_counts.max()}")

# ─────────────────────────────────────────────────────────────────────────────
# REASON / STATUS / HUB BREAKDOWNS
# Note: "Change Request" holds the reason category; "Change Request
# Description" holds the free-text explanation (these are easy to mix up).
# ─────────────────────────────────────────────────────────────────────────────
total = len(rematch)

print("\n=== Reason breakdown ===")
print(rematch['Change Request'].value_counts())
print((100 * rematch['Change Request'].value_counts() / total).round(1))

print("\n=== Status breakdown ===")
print(rematch['Change Request Status'].value_counts(dropna=False))
print((100 * rematch['Change Request Status'].value_counts(dropna=False) / total).round(1))

print("\n=== By hub ===")
print(rematch['Hubs (from Students)'].value_counts())
print((100 * rematch['Hubs (from Students)'].value_counts() / total).round(1))

# Rematch rate relative to each hub's matched student count (from the
# established placement history pipeline)
hub_matched_totals = {
    'Community and Social Services': 445,
    'Healthcare': 468,
    'Marketing and Communications': 457,
    'STEM and Green': 446,
}
print("\n=== Rematch rate by hub (requests / matched students in that hub) ===")
for hub, hub_total in hub_matched_totals.items():
    n = (rematch['Hubs (from Students)'] == hub).sum()
    print(f"{hub}: {n} requests / {hub_total} matched = {100 * n / hub_total:.1f}%")

rematch.to_csv(data_path('rematch_requests_clean.csv'), index=False)